# 🎬 Ai-mation — توليد فيديو مجاني على Google Colab (من الهاتف)

يعمل بنموذج **LTX-Video 0.9.5** المفتوح على كرت شاشة **T4 المجاني** من Google.

## الخطوات
1. من القائمة: **Runtime ← Change runtime type ← T4 GPU ← Save**
   (على الهاتف: اضغط ⋮ أو اسم الملف ← Runtime)
2. شغّل الخلايا **بالترتيب** (1️⃣ ثم 2️⃣ ثم 3️⃣) بالضغط على ▶️ بجانب كل خلية.
3. الخلية 1 و2 تأخذ 5-10 دقائق في كل جلسة جديدة (تحميل النموذج ~25GB على سيرفر Google، وليس على هاتفك). بعدها كل فيديو يأخذ دقائق قليلة.
4. **🎞️ فيديو طويل**: عدة مشاهد متصلة في فيديو واحد.
5. **🎙️ تعليق صوتي**: صوت عربي (عراقي، سعودي، مصري، إماراتي) + ترجمة مكتوبة على الفيديو.
6. غيّر الوصف في الخلية 3 واضغط ▶️ لتوليد فيديو جديد — بدون إعادة الخلايا 1 و2.

> ⚠️ الجلسة المجانية محدودة بساعات يومياً، وقد لا يتوفر T4 أوقات الذروة. إذا أُغلقت الجلسة أعد تشغيل الخلايا من البداية.

## 💡 سر النتيجة الجيدة
- **وصف طويل ومفصل**: الشكل، الملابس، المكان، الحركة بالضبط، الإضاءة.
- **حركة واحدة بسيطة** في كل مشهد (تمشي، تلتفت، تبتسم) بدل أحداث كثيرة.
- **الكاميرا الثابتة** تعطي أقل تشوه.
- إذا أعجبك مشهد احفظ رقم **seed** وأعد استخدامه.


In [ ]:
#@title 1️⃣ تجهيز البيئة (مرة واحدة)
import subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("❌ لا يوجد GPU. اذهب إلى Runtime ← Change runtime type ← اختر T4 GPU ثم أعد التشغيل.")
print("✅ GPU:", gpu)
# نسخة torchao المثبتة مسبقاً في Colab لا تتوافق مع diffusers، ولا نحتاجها
!pip uninstall -y -q torchao
!pip install -q diffusers==0.41.0 "bitsandbytes>=0.48.0" accelerate sentencepiece imageio imageio-ffmpeg pillow-heif
print("✅ تم التثبيت")

In [ ]:
#@title 2️⃣ تحميل النموذج (مرة واحدة لكل جلسة)
#@markdown إذا خرج الفيديو أسود أو مشوّه، غيّر الدقة إلى bfloat16 وأعد تشغيل هذه الخلية.
precision = "float16" #@param ["float16", "bfloat16"]

import torch, gc
from transformers import T5EncoderModel, BitsAndBytesConfig
from diffusers import LTXConditionPipeline

# الإصدار 0.9.5: حركة أكثر ثباتاً وأفضل بكثير في تحريك الصور وإكمال المشاهد
MODEL = "Lightricks/LTX-Video-0.9.5"
# نسخة bf16 من نفس مُرمّز النص (T5-XXL): 9.5GB بدل 19GB
TEXT_ENCODER = "city96/t5-v1_1-xxl-encoder-bf16"
dtype = torch.float16 if precision == "float16" else torch.bfloat16

# إذا بقيت ذاكرة الكرت محجوزة من محاولة سابقة فاشلة، يجب إعادة تشغيل الجلسة
free, total = torch.cuda.mem_get_info()
if total - free > 2e9:
    raise SystemExit(
        f"⚠️ ذاكرة الكرت مشغولة ({(total - free) / 1e9:.1f} GB) من محاولة سابقة.\n"
        "اضغط Runtime ← Restart session، ثم شغّل الخلية 1️⃣ ثم هذه الخلية من جديد."
    )

print("⏳ (1/2) تحميل مُرمّز النص... قد يستغرق عدة دقائق")
# 4-bit يقلل استهلاك ذاكرة الكرت إلى ~3GB، وbfloat16 يمنع أخطاء T5 التي تحدث مع float16
text_encoder = T5EncoderModel.from_pretrained(
    TEXT_ENCODER,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16,
    ),
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)
gc.collect(); torch.cuda.empty_cache()

print("⏳ (2/2) تحميل نموذج الفيديو...")
# خط معالجة واحد للنص والصورة وإكمال الفيديو
pipe = LTXConditionPipeline.from_pretrained(MODEL, text_encoder=text_encoder, torch_dtype=dtype, low_cpu_mem_usage=True)
pipe.transformer.to("cuda")
pipe.vae.to("cuda")
pipe.vae.enable_tiling()
i2v = pipe
gc.collect(); torch.cuda.empty_cache()
# ---------- أدوات التوليد ----------
import os, re, time, random, urllib.parse, urllib.request, json
from diffusers.utils import export_to_video
from IPython.display import HTML, display
from base64 import b64encode

os.makedirs("/content/videos", exist_ok=True)
SIZES = {"طولي 9:16 (تيك توك/ريلز)": (768, 512), "عرضي 16:9 (يوتيوب)": (512, 768), "مربع 1:1": (640, 640)}
NEGATIVE = "worst quality, inconsistent motion, blurry, jittery, distorted, deformed, morphing, warping, flickering, chaotic movement, extra limbs, watermark, text, nsfw, nudity"
# يضاف لكل وصف: النموذج يتحرك عشوائياً إذا كان الوصف قصيراً ولا يحدد الكاميرا
CAMERA = {
    "ثابتة": "The camera remains stationary.",
    "تقترب ببطء": "The camera slowly pushes in.",
    "تبتعد ببطء": "The camera slowly pulls back.",
    "تتبع الشخصية": "The camera smoothly follows the subject.",
    "تتحرك أفقياً": "The camera pans slowly from left to right.",
}
QUALITY = "Smooth, natural and consistent motion. The scene is captured in real-life footage, highly detailed."

def enrich(prompt_en, camera="ثابتة"):
    return f"{prompt_en.rstrip('. ')}. {CAMERA.get(camera, '')} {QUALITY}"
BLOCKED = ["nude", "naked", "nsfw", "porn", "sex", "erotic", "hentai", "topless",
           "عاري", "عارية", "جنس", "جنسي", "إباحي", "اباحي", "سكس", "بدون ملابس"]

def translate(text):
    if not re.search(r"[\u0600-\u06FF]", text):
        return text
    try:
        url = "https://api.mymemory.translated.net/get?langpair=ar|en&q=" + urllib.parse.quote(text[:500])
        t = json.load(urllib.request.urlopen(url, timeout=20))["responseData"]["translatedText"]
        print("🌐 الترجمة:", t)
        return t
    except Exception as e:
        print("⚠️ تعذرت الترجمة، سيُستخدم النص كما هو:", e)
        return text

def check(text):
    low = " " + re.sub(r"[^\w]+", " ", text.lower()) + " "
    if any(f" {w} " in low for w in BLOCKED):
        raise SystemExit("❌ الوصف يحتوي محتوى جنسياً صريحاً، وهذا غير مدعوم.")

def frames_for(seconds):
    n = int(seconds * 24)
    return (n // 8) * 8 + 1  # النموذج يحتاج عدد إطارات = 8k+1

def show(path):
    data = b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="100%" controls autoplay loop playsinline src="data:video/mp4;base64,{data}"></video>'))
    print("💾 محفوظ في:", path, "— لتحميله: افتح أيقونة 📁 على اليسار، ثم ⋮ بجانب الملف ← Download")

def generate(pipeline, prompt_en, h, w, seconds, steps, seed, **extra):
    """يولّد مشهداً واحداً ويرجع قائمة الإطارات"""
    # نرمّز النص بأنفسنا ونحوّله إلى دقة نموذج الفيديو
    with torch.no_grad():
        pe, pm, ne, nm = pipeline.encode_prompt(
            prompt=prompt_en, negative_prompt=NEGATIVE, do_classifier_free_guidance=True,
            max_sequence_length=256, device="cuda", dtype=dtype,
        )
    frames = pipeline(
        prompt_embeds=pe, prompt_attention_mask=pm,
        negative_prompt_embeds=ne, negative_prompt_attention_mask=nm,
        width=w, height=h, num_frames=frames_for(seconds), frame_rate=24,
        num_inference_steps=steps, guidance_scale=3.0,
        decode_timestep=0.05, decode_noise_scale=0.025,
        generator=torch.Generator("cuda").manual_seed(seed), **extra,
    ).frames[0]
    gc.collect(); torch.cuda.empty_cache()
    return frames

def save_frames(frames, name="aimation"):
    path = f"/content/videos/{name}_{int(time.time())}.mp4"
    export_to_video(frames, path, fps=24)
    return path

def run(pipeline, prompt, size, seconds, steps, seed, camera="ثابتة", **extra):
    check(prompt)
    prompt_en = enrich(translate(prompt), camera)
    check(prompt_en)
    h, w = SIZES[size]
    seed = seed if seed >= 0 else random.randint(0, 2**31 - 1)
    print(f"🎬 جارٍ التوليد... ({seconds} ثوانٍ، {steps} خطوة، seed={seed})")
    t0 = time.time()
    frames = generate(pipeline, prompt_en, h, w, seconds, steps, seed, **extra)
    path = save_frames(frames)
    print(f"✅ تم في {time.time() - t0:.0f} ثانية")
    show(path)
    return path

print("✅ جاهز")

print(f"✅ النموذج جاهز — ذاكرة الكرت المستخدمة: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


In [ ]:
#@title 3️⃣ نص ← فيديو
#@markdown اكتب وصفاً **مفصلاً**: من الشخصية وشكلها وملابسها، أين هي، ماذا تفعل بالضبط، الإضاءة. الوصف القصير = حركة عشوائية.
prompt = "محارب صحراوي يمشي ببطء أمام قلعة قديمة عند الغروب، الرياح تحرك عباءته، الكاميرا تتبعه بهدوء، إضاءة سينمائية" #@param {type:"string"}
size = "طولي 9:16 (تيك توك/ريلز)" #@param ["طولي 9:16 (تيك توك/ريلز)", "عرضي 16:9 (يوتيوب)", "مربع 1:1"]
seconds = 4 #@param {type:"slider", min:2, max:6, step:1}
steps = 30 #@param {type:"slider", min:15, max:50, step:5}
seed = -1 #@param {type:"integer"}
camera = "ثابتة" #@param ["ثابتة", "تقترب ببطء", "تبتعد ببطء", "تتبع الشخصية", "تتحرك أفقياً"]
#@markdown ↑ الكاميرا الثابتة تعطي أقل تشوه
#@markdown seed = -1 يعني عشوائي. استخدم نفس الرقم لتكرار نفس النتيجة.

if "run" not in globals() or "pipe" not in globals():
    raise SystemExit("⚠️ شغّل الخلية 2️⃣ أولاً وانتظر رسالة ✅ النموذج جاهز، ثم شغّل هذه الخلية.")
video_path = run(pipe, prompt, size, seconds, steps, seed, camera)

In [ ]:
#@title 4️⃣ صورة ← فيديو (تحريك صورة)
#@markdown **طريقة رفع الصورة (الأسهل من الهاتف):** ☰ ← **Show file browser** ← أيقونة الرفع ⬆️ ← اختر الصورة. انتظر حتى تظهر في القائمة، ثم شغّل هذه الخلية: ستُستخدم **أحدث صورة رفعتها** تلقائياً.
#@markdown أو الصق رابط صورة من الإنترنت في خانة image بدلاً من ذلك.
motion = "الشخصية تبتسم وتلتفت ببطء نحو الكاميرا، الشعر يتحرك مع النسيم" #@param {type:"string"}
size = "طولي 9:16 (تيك توك/ريلز)" #@param ["طولي 9:16 (تيك توك/ريلز)", "عرضي 16:9 (يوتيوب)", "مربع 1:1"]
seconds = 4 #@param {type:"slider", min:2, max:6, step:1}
steps = 30 #@param {type:"slider", min:15, max:50, step:5}
seed = -1 #@param {type:"integer"}
camera = "ثابتة" #@param ["ثابتة", "تقترب ببطء", "تبتعد ببطء", "تتبع الشخصية", "تتحرك أفقياً"]
#@markdown ↑ الكاميرا الثابتة تعطي أقل تشوه
image = "" #@param {type:"string"}
#@markdown ↑ اتركه فارغاً لاستخدام أحدث صورة مرفوعة، أو اكتب اسم الملف، أو الصق رابط صورة (https://...)

if "run" not in globals() or "pipe" not in globals():
    raise SystemExit("⚠️ شغّل الخلية 2️⃣ أولاً وانتظر رسالة ✅ النموذج جاهز، ثم شغّل هذه الخلية.")

import glob, io
from PIL import Image, ImageOps
try:  # صور HEIC من بعض الهواتف
    from pillow_heif import register_heif_opener
    register_heif_opener()
except ImportError:
    pass

IMG_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".heic")
src = image.strip()
if src.startswith("http"):
    print("🌐 تحميل الصورة من الرابط...")
    req = urllib.request.Request(src, headers={"User-Agent": "Mozilla/5.0"})
    img = Image.open(io.BytesIO(urllib.request.urlopen(req, timeout=30).read()))
    name = src
else:
    if src:
        candidates = [src if src.startswith("/") else f"/content/{src}"]
    else:
        candidates = sorted(
            (f for f in glob.glob("/content/*") + glob.glob("/content/drive/MyDrive/*")
             if f.lower().endswith(IMG_EXT)),
            key=os.path.getmtime,
        )[-1:]
    if not candidates or not os.path.exists(candidates[0]):
        raise SystemExit(
            "❌ لم أجد صورة.\n"
            "ارفعها من: ☰ ← Show file browser ← أيقونة الرفع ⬆️، وانتظر ظهورها في القائمة، ثم أعد تشغيل الخلية.\n"
            "أو الصق رابط صورة في خانة image."
        )
    name = candidates[0]
    img = Image.open(name)
print("🖼️ الصورة:", os.path.basename(name))

h, w = SIZES[size]
img = ImageOps.fit(ImageOps.exif_transpose(img).convert("RGB"), (w, h))
display(img.resize((w // 3, h // 3)))
video_path = run(i2v, motion, size, seconds, steps, seed, camera, image=img)


In [ ]:
#@title 🎞️ فيديو طويل (عدة مشاهد متصلة)
#@markdown اكتب المشاهد مفصولة بعلامة **|** . اكتب في خانة **character** وصف الشخصية والمكان مرة واحدة ليُضاف لكل مشهد فتبقى متطابقة.
scenes = "فتاة تمشي في سوق شعبي قديم نهاراً | الفتاة تتوقف أمام بائع التوابل وتبتسم | الكاميرا ترتفع ببطء لتظهر السوق كله من الأعلى" #@param {type:"string"}
character = "فتاة شابة بشعر أسود طويل ترتدي فستاناً أحمر، في سوق شعبي مزدحم بالبضائع الملونة" #@param {type:"string"}
style = "سينمائي، إضاءة دافئة" #@param {type:"string"}
#@markdown ↑ نمط يُضاف لكل المشاهد ليبقى الشكل موحداً (اختياري)
size = "طولي 9:16 (تيك توك/ريلز)" #@param ["طولي 9:16 (تيك توك/ريلز)", "عرضي 16:9 (يوتيوب)", "مربع 1:1"]
seconds_per_scene = 4 #@param {type:"slider", min:2, max:6, step:1}
steps = 30 #@param {type:"slider", min:15, max:50, step:5}
transition = "مونتاج (قطع بين المشاهد)" #@param ["مونتاج (قطع بين المشاهد)", "متصل (إكمال من آخر المشهد)"]
#@markdown ↑ **المونتاج** أوضح وأقل تشوهاً (مثل الأفلام). **المتصل** حركة واحدة مستمرة لكن قد تتراكم الأخطاء.
camera = "ثابتة" #@param ["ثابتة", "تقترب ببطء", "تبتعد ببطء", "تتبع الشخصية", "تتحرك أفقياً"]
seed = -1 #@param {type:"integer"}

if "run" not in globals() or "pipe" not in globals():
    raise SystemExit("⚠️ شغّل الخلية 2️⃣ أولاً وانتظر رسالة ✅ النموذج جاهز، ثم شغّل هذه الخلية.")

parts = [x.strip() for x in scenes.split("|") if x.strip()]
if not parts:
    raise SystemExit("اكتب مشهداً واحداً على الأقل.")
check(scenes + " " + style + " " + character)
style_en = translate(style) if style.strip() else ""
char_en = translate(character) if character.strip() else ""
h, w = SIZES[size]
seed = seed if seed >= 0 else random.randint(0, 2**31 - 1)
print(f"🎞️ {len(parts)} مشاهد × {seconds_per_scene} ثوانٍ ≈ {len(parts) * seconds_per_scene} ثانية (seed={seed})")

all_frames, t0 = [], time.time()
for i, scene in enumerate(parts):
    prompt_en = enrich(". ".join(x for x in [char_en, translate(scene), style_en] if x), camera)
    check(prompt_en)
    print(f"\n▶️ المشهد {i + 1}/{len(parts)}: {prompt_en}")
    if i > 0 and transition.startswith("متصل"):
        # نكمل من آخر 9 إطارات (وليس صورة واحدة) لتستمر الحركة بسلاسة
        tail = all_frames[-9:]
        frames = generate(pipe, prompt_en, h, w, seconds_per_scene, steps, seed + i, video=tail)
        frames = frames[len(tail):]  # بداية المقطع الجديد مكررة من المشهد السابق
    else:
        frames = generate(pipe, prompt_en, h, w, seconds_per_scene, steps, seed + i)
    all_frames.extend(frames)

video_path = save_frames(all_frames, "aimation_long")
print(f"\n✅ تم في {(time.time() - t0) / 60:.1f} دقيقة")
show(video_path)

In [ ]:
#@title 🎙️ تعليق صوتي عربي + ترجمة على الفيديو
#@markdown يضيف صوتاً عربياً طبيعياً لآخر فيديو ولّدته (من أي خلية). إذا كان الصوت أطول من الفيديو يتكرر الفيديو تلقائياً.
narration = "في قلب المدينة القديمة، حيث تحكي الجدران قصص الأجداد… بدأت الحكاية." #@param {type:"string"}
voice = "عراقي - رجل (باسل)" #@param ["عراقي - رجل (باسل)", "عراقي - امرأة (رنا)", "سعودي - رجل (حامد)", "سعودية - امرأة (زارية)", "مصري - رجل (شاكر)", "مصرية - امرأة (سلمى)", "إماراتي - رجل (حمدان)", "إماراتية - امرأة (فاطمة)"]
speed = 0 #@param {type:"slider", min:-30, max:30, step:5}
#@markdown ↑ سرعة الكلام بالنسبة المئوية (0 = عادي)
subtitles = True #@param {type:"boolean"}
video_file = "" #@param {type:"string"}
#@markdown ↑ اتركه فارغاً لاستخدام آخر فيديو، أو اكتب اسم ملف من مجلد videos

!pip install -q edge-tts==7.3.1
import edge_tts, glob, subprocess, json as _json

VOICES = {
    "عراقي - رجل (باسل)": "ar-IQ-BasselNeural", "عراقي - امرأة (رنا)": "ar-IQ-RanaNeural",
    "سعودي - رجل (حامد)": "ar-SA-HamedNeural", "سعودية - امرأة (زارية)": "ar-SA-ZariyahNeural",
    "مصري - رجل (شاكر)": "ar-EG-ShakirNeural", "مصرية - امرأة (سلمى)": "ar-EG-SalmaNeural",
    "إماراتي - رجل (حمدان)": "ar-AE-HamdanNeural", "إماراتية - امرأة (فاطمة)": "ar-AE-FatimaNeural",
}

if not narration.strip():
    raise SystemExit("اكتب نص التعليق أولاً.")
check(narration)

if video_file.strip():
    src = video_file if video_file.startswith("/") else f"/content/videos/{video_file.strip()}"
else:
    vids = sorted((v for v in glob.glob("/content/videos/*.mp4") if "_voice" not in v), key=os.path.getmtime)
    if not vids:
        raise SystemExit("لا يوجد فيديو بعد. ولّد فيديو من الخلية 3️⃣ أو 4️⃣ أو 🎞️ أولاً.")
    src = vids[-1]
print("🎬 الفيديو:", os.path.basename(src))

# 1) توليد الصوت والترجمة
base = src[:-4]
audio, srt = base + "_voice.mp3", base + ".srt"
communicate = edge_tts.Communicate(narration, VOICES[voice], rate=f"{speed:+d}%")
submaker = edge_tts.SubMaker()
with open(audio, "wb") as f:
    async for chunk in communicate.stream():
        if chunk["type"] == "audio":
            f.write(chunk["data"])
        elif chunk["type"] in ("WordBoundary", "SentenceBoundary"):
            submaker.feed(chunk)
open(srt, "w", encoding="utf-8").write(submaker.get_srt())
print("🎙️ تم توليد الصوت")

def duration(path):
    out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "json", path],
                         capture_output=True, text=True).stdout
    return float(_json.loads(out)["format"]["duration"])

v_len, a_len = duration(src), duration(audio)
print(f"⏱️ الفيديو {v_len:.1f}ث — الصوت {a_len:.1f}ث")

# 2) خط عربي للترجمة
vf = []
if subtitles:
    font_dir = "/content/fonts"
    font = f"{font_dir}/NotoSansArabic.ttf"
    if not os.path.exists(font):
        os.makedirs(font_dir, exist_ok=True)
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/google/fonts/main/ofl/notosansarabic/NotoSansArabic%5Bwdth,wght%5D.ttf", font)
        except Exception as e:
            print("⚠️ تعذر تحميل الخط، سيتم الدمج بدون ترجمة:", e)
    if os.path.exists(font):
        sub_style = "FontName=Noto Sans Arabic,Bold=1,FontSize=13,PrimaryColour=&H00FFFFFF,OutlineColour=&H00000000,BorderStyle=1,Outline=2,Shadow=0,MarginV=40,Alignment=2"
        vf = ["-vf", f"subtitles={srt}:fontsdir={font_dir}:force_style='{sub_style}'"]

# 3) الدمج
out = base + "_voice.mp4"
cmd = ["ffmpeg", "-y", "-loglevel", "error"]
if a_len > v_len:
    cmd += ["-stream_loop", "-1"]  # نكرر الفيديو حتى ينتهي الصوت
cmd += ["-i", src, "-i", audio, *vf, "-map", "0:v", "-map", "1:a",
        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "128k"]
if a_len > v_len:
    cmd += ["-shortest"]
cmd += [out]
r = subprocess.run(cmd, capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("❌ فشل الدمج:\n" + r.stderr[-1500:])
video_path = out
print("✅ تم إضافة الصوت" + (" والترجمة" if vf else ""))
show(out)

In [ ]:
#@title 5️⃣ (اختياري) حفظ كل الفيديوهات في Google Drive
from google.colab import drive
import shutil
drive.mount("/content/drive")
dest = "/content/drive/MyDrive/Ai-mation"
os.makedirs(dest, exist_ok=True)
for f in os.listdir("/content/videos"):
    shutil.copy(f"/content/videos/{f}", dest)
print("✅ تم الحفظ في Google Drive ← مجلد Ai-mation")